# [INFO] Glu-Stock: 03_EXECUTION_MONITOR
**Phase**: Institutional Daily Recap Hub (v18.5)

This unified notebook manages daily trades and sends a comprehensive 08:30 WIB recap of Yesterday's Results and Today's Targets.

In [ ]:
# [INSTALL] SECTION 1: INSTALLATION
!pip install -q yfinance firebase-admin pandas pyTelegramBotAPI ta psutil python-dotenv


In [ ]:
# [INFO] SECTION 2: INFRASTRUCTURE (Firebase, Telegram & Secrets)
import json, os, firebase_admin, joblib, numpy as np, pandas as pd, yfinance as yf, warnings, psutil, time
from firebase_admin import credentials, firestore
from datetime import datetime, timedelta
import telebot, ta
warnings.filterwarnings('ignore')

try:
    from kaggle_secrets import UserSecretsClient
    IS_KAGGLE = True
except ImportError:
    IS_KAGGLE = False

class KaggleInfra:
    @staticmethod
    def load_secrets():
        if IS_KAGGLE:
            user_secrets = UserSecretsClient()
            try:
                raw_firebase = user_secrets.get_secret("FIREBASE_KEY_JSON")
                token = user_secrets.get_secret("TELEGRAM_TOKEN")
                chat_id = user_secrets.get_secret("TELEGRAM_CHAT_ID")
                return {"key": json.loads(raw_firebase), "token": token, "chat_id": chat_id}
            except Exception as e:
                return {"key": None, "token": None, "chat_id": None}
        else:
            from dotenv import load_dotenv
            load_dotenv()
            return {
                "key": json.loads(os.getenv("FIREBASE_KEY_JSON", "{}")),
                "token": os.getenv("TELEGRAM_TOKEN"),
                "chat_id": os.getenv("TELEGRAM_CHAT_ID")
            }

class FirebaseHandler:
    def __init__(self, secrets):
        if not firebase_admin._apps:
            if not secrets.get('key'):
                raise ValueError("FIREBASE_KEY_JSON is missing.")
            cred = credentials.Certificate(secrets['key'])
            firebase_admin.initialize_app(cred)
        self.db = firestore.client()

    def wait_for_queue(self, queue_name: str, max_retries=15, interval=60):
        for i in range(max_retries):
            docs = self.db.collection(f"glu_stock_queue_{queue_name}").get()
            if docs:
                tasks = []
                for doc in docs:
                    dt = doc.to_dict()
                    tasks.append(dt.get('payload', dt))
                    doc.reference.delete()
                return tasks
            if i < max_retries - 1:
                print(f"[WAIT] {queue_name} queue empty. Retrying...", flush=True)
                time.sleep(interval)
        return []
        
    def get_closed_trades_24h(self):
        yesterday = (datetime.now() - timedelta(days=1)).isoformat()
        docs = self.db.collection("glu_stock_trades").where("status", "==", "CLOSED").where("exit_date", ">", yesterday).get()
        return [doc.to_dict() for doc in docs]

    def get_active_trades(self):
        docs = self.db.collection("glu_stock_trades").where("status", "==", "OPEN").get()
        return [(doc.id, doc.to_dict()) for doc in docs]
        
    def update_trade(self, doc_id, data):
        self.db.collection("glu_stock_trades").document(doc_id).update(data)
        
    def insert_trade(self, trade_data):
        self.db.collection("glu_stock_trades").add(trade_data)
        
    def log_event(self, phase, details):
        self.db.collection("glu_stock_history").add({
            'timestamp': datetime.now().isoformat(), 
            'phase': phase.upper(), 
            'details': details
        })


In [ ]:
# [LOGIC] SECTION 3: RECAP AGENT
class RiskManager:
    def calculate_position(self, df, ticker, equity=100000000):
        try:
            close = df['Close'].squeeze()
            curr_price = float(close.iloc[-1])
            atr = ta.volatility.AverageTrueRange(high=df['High'].squeeze(), low=df['Low'].squeeze(), close=close, window=14).average_true_range().iloc[-1]
            sl = curr_price - (2.0 * atr)
            risk_per_share = curr_price - sl
            shares = int((equity * 0.01) / (risk_per_share + 1e-7))
            return shares, sl, curr_price + (3.0 * atr)
        except: return 0, 0, 0

class TradingAgent:
    def __init__(self, fb):
        self.fb = fb
        self.risk = RiskManager()
        self.today_logs = []

    def _log(self, phase, msg):
        self.today_logs.append(f"[{phase}] {msg}")
        self.fb.log_event(phase, msg)
        print(f"[{phase}] {msg}")

    def manage_open_trades(self):
        active = self.fb.get_active_trades()
        for doc_id, t in active:
            try:
                df = yf.download(t['ticker'], period='1d', progress=False)
                curr = float(df['Close'].iloc[-1])
                pnl = (curr - t['entry_price']) * t['shares']
                if curr <= t['stop_loss']: 
                    self.fb.update_trade(doc_id, {'status': 'CLOSED', 'exit_price': curr, 'reason': 'SL', 'pnl': pnl, 'exit_date': datetime.now().isoformat()})
                    self._log('CLOSER', f"Sold {t['ticker']} (SL Hit)")
                elif curr >= t['take_profit']:
                    self.fb.update_trade(doc_id, {'status': 'CLOSED', 'exit_price': curr, 'reason': 'TP', 'pnl': pnl, 'exit_date': datetime.now().isoformat()})
                    self._log('CLOSER', f"Sold {t['ticker']} (TP Hit)")
            except: pass

    def execute_signals(self, signals):
        for ticker, data in signals.items():
            try:
                existing = [t[1]['ticker'] for t in self.fb.get_active_trades()]
                if ticker in existing: continue
                if len(existing) >= 10: break
                df = yf.download(ticker, period='30d', progress=False)
                shares, sl, tp = self.risk.calculate_position(df, ticker)
                if shares > 0:
                    self.fb.insert_trade({'ticker': ticker, 'shares': shares, 'entry_price': data['price'], 'stop_loss': sl, 'take_profit': tp, 'status': 'OPEN', 'entry_date': datetime.now().isoformat()})
                    self._log('BUY', f"Bought {ticker} @ {data['price']:,.0f}")
            except: pass


In [ ]:
# [HUB] SECTION 4: 08:30 WIB SYNC & RECAP CYCLE
def wait_until_0830_wib():
    # UTC+7 for WIB
    now_utc = datetime.utcnow()
    now_wib = now_utc + timedelta(hours=7)
    target_wib = now_wib.replace(hour=8, minute=30, second=0, microsecond=0)
    
    if now_wib > target_wib:
        print(f"[INFO] Already past 08:30 WIB ({now_wib.strftime('%H:%M')}). Sending report now.")
        return
        
    diff = (target_wib - now_wib).total_seconds()
    print(f"[WAIT] Syncing with market opening. Sleeping for {diff/60:.1f} minutes until 08:30 WIB...", flush=True)
    # Split into chunks to keep notebook alive if needed
    chunk = 600 # 10 mins
    while diff > 0:
        slp = min(diff, chunk)
        time.sleep(slp)
        diff -= slp
        print(f"[TICK] {diff/60:.1f} mins remaining...", flush=True)

def run_recap_hub():
    secrets = KaggleInfra.load_secrets()
    fb = FirebaseHandler(secrets)
    agent = TradingAgent(fb)
    
    # 1. Action (Run ASAP after trigger)
    agent.manage_open_trades()
    signals = fb.wait_for_queue('signals')
    if signals:
        all_sig = {}
        for s in signals: all_sig.update(s)
        agent.execute_signals(all_sig)
    
    # 2. Wait for 08:30 WIB
    if IS_KAGGLE: wait_until_0830_wib()
    
    # 3. Build Detailed Report
    closed_24h = fb.get_closed_trades_24h()
    active = fb.get_active_trades()
    
    recap_str = "\n".join([f"- {c['ticker']}: {c['pnl']:+,.0f} ({c['reason']})" for c in closed_24h])
    target_str = "\n".join(agent.today_logs) if agent.today_logs else "- No targets met filters."
    port_str = ""
    total_unrealized = 0
    for _, t in active:
        try:
            df = yf.download(t['ticker'], period='1d', progress=False)
            curr = float(df['Close'].iloc[-1])
            pnl = (curr - t['entry_price']) * t['shares']
            total_unrealized += pnl
            port_str += f"- {t['ticker']}: {curr:,.0f} ({pnl:+,.0f})\n"
        except: pass
        
    report = (
        f"**[GLU-STOCK] DAILY RECAP HUB**\n"
        f"**WKT**: {datetime.utcnow()+timedelta(hours=7):%Y-%m-%d %H:%M} WIB\n\n"
        f"**[1] REKAP KEMARIN**:\n{recap_str if recap_str else '- No trades closed.'}\n\n"
        f"**[2] TARGET HARI INI**:\n{target_str}\n\n"
        f"**[3] PORTO AKTIF**:\n{port_str if port_str else '- Empty'}\n\n"
        f"**TOTAL UNREALIZED**: {total_unrealized:+,.0f} IDR\n"
        f"**SYSTEM**: RAM {psutil.virtual_memory().percent}%"
    )
    
    if secrets.get('token'):
        try:
            bot = telebot.TeleBot(secrets['token'])
            bot.send_message(secrets['chat_id'], report, parse_mode="Markdown")
            print("[OK] Recap sent at 08:30 WIB.")
        except Exception as e: print(f"Error sending: {e}")

run_recap_hub()